In [ ]:
Software Engineering 458 - Ratt Gang Capstone Project

    by Marley Thrift, Audrey Fellows, Ryan Dube and William Ervin

    
Description: Sequential image processing using pytorch libraries and d2l (deep learning)

Import/Install

In [ ]:
# install setuptools and d2l, required for running d2l in this environment
# May cause you to restart your runtime
!pip install setuptools==84.0.0 wheel==0.48.0
!pip install d2l==1.0.3


In [ ]:
# Import Libraries
import os
import csv
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torchvision.models import resnet18, ResNet18_Weights
from torchcodec.decoders import VideoDecoder
import cv2


Load Data

In [ ]:
# import drives if necessary

In [ ]:
# path data to variables
train_path = './'
test_path = './'

Preprocessing

In [ ]:
# import videos, create path to accessable data, 
# research will be needed on how to process videos into sequential imaging

### I have come across the tool torchcodec for decrypting video files into tensors. You will have to run it with a !pip install, perhaps after the torch import - Marley ###
# !pip install torch torchcodec ### not sure if needed may be in imports already

#Input each video and output the timestamps where motion is detected

def find_motion(video_path, min_pixel_change=800, video_pad=2.0, frame_merge_gap=3.0):
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0 #If fps is unaviable, default to 30.0
        background = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=25)

        moving_time_stamp = []
        frame_number = 0

        while True:
            ok, frame = cap.read()
            if not ok:
                break
            smaller_res = cv2.resize(frame, None, fx=0.5, fy=0.5)
            mask = background.apply(smaller_res)
            mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, None, iterations=2)
            if cv2.countNonZero(mask) > min_pixel_change:
                 moving_time_stamp.append(frame_number / fps)
            frame_number += 1
        cap.release()

        windowed_time_stamps = []
        for t in moving_time_stamp:
             if windowed_time_stamps and t - windowed_time_stamps[-1][1] <= frame_merge_gap:
                  windowed_time_stamps[-1][1] = t
             else:
                  windowed_time_stamps.append([t, t])

        return [(max(0, start - video_pad), end + video_pad) for start, end in windowed_time_stamps]

windowed_time_stamps = find_motion(video_path)
for start, end in windowed_time_stamps:
     print(f"{start:.1f}-{end:.1f}")





In [ ]:
decoder = VideoDecoder(
    video_path,
    dimension_order="NCHW",  # default config "num_frames, channel, height, width"
)

frame_batch = decoder.get_frames_at(indices=frame_indices)
video = frame_batch.data

print(video.shape)
# [num_frames, 3, height, width]
# __getitem__ should return video, label

In [ ]:
# VideoDecoder → [frames, channels, height, width]
# DataLoader   → [batch, frames, channels, height, width]
# Model        → [batch] logits

In [ ]:
class VideoDataset(Dataset):
    def __init__(self, video_paths, labels, num_frames=16):
        self.video_paths = video_paths
        self.labels = labels
        self.num_frames = num_frames

        # Resizes, crops, scales, and normalizes frames for ResNet-18
        self.transform = ResNet18_Weights.DEFAULT.transforms()

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, index):
        video_path = self.video_paths[index]
        label = self.labels[index]

        decoder = VideoDecoder(
            video_path,
            dimension_order="NCHW",
        )

        total_frames = len(decoder)

        # Uniformly sample a fixed number of frames.
        frame_indices = torch.linspace(
            0,
            total_frames - 1,
            steps=self.num_frames,
        ).round().long()

        # Shape: [num_frames, channels, height, width]
        frames = decoder.get_frames_at(
            indices=frame_indices.tolist()
        ).data

        # Applies ResNet's required image preprocessing.
        # Shape remains [num_frames, 3, 224, 224].
        frames = self.transform(frames)

        # BCEWithLogitsLoss expects floating-point labels.
        label = torch.tensor(label, dtype=torch.float32)

        return frames, label

In [ ]:
dataset = VideoDataset(
    video_paths=video_paths,
    labels=labels,
    num_frames=16,
)

loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True,
    num_workers=4,
)

videos, labels = next(iter(loader))

print(videos.shape)  # [8, 16, 3, 224, 224]
print(labels.shape)  # [8]

__Fully Connected Neural Network__

Classifiers

In [ ]:
class BinaryVideoClassifier(nn.Module):
    def __init__(
        self,
        hidden_dim=256,
        num_lstm_layers=1,
        dropout=0.3,
    ):
        super().__init__()

        # CNN that extracts features from each frame.
        # weights=None avoids downloading pretrained weights for this test.
        backbone = resnet18(weights=None)

        # ResNet-18 normally sends its 512 features into its own classifier.
        feature_dim = backbone.fc.in_features  # 512

        # Remove ResNet's original classifier because we want its features.
        backbone.fc = nn.Identity()
        self.frame_encoder = backbone

        # Processes the sequence of frame features.
        self.temporal_model = nn.LSTM(
            input_size=feature_dim,
            hidden_size=hidden_dim,
            num_layers=num_lstm_layers,
            batch_first=True,
            dropout=dropout if num_lstm_layers > 1 else 0.0,
        )

        # Converts each video's features into one binary-classification score.
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, videos):
        """
        videos shape:
        [batch, frames, channels, height, width]
        """

        batch_size, num_frames, channels, height, width = videos.shape

        # Change:
        # [batch, frames, channels, height, width]
        # into:
        # [batch * frames, channels, height, width]
        #
        # This allows ResNet to process every frame as an image.
        frames = videos.reshape(
            batch_size * num_frames,
            channels,
            height,
            width,
        )

        # Extract 512 features from each frame.
        frame_features = self.frame_encoder(frames)

        # Restore the batch and frame dimensions:
        # [batch, frames, 512]
        frame_features = frame_features.reshape(
            batch_size,
            num_frames,
            -1,
        )

        # Process the frame sequence with the LSTM.
        # Shape: [batch, frames, hidden_dim]
        temporal_features, _ = self.temporal_model(frame_features)

        # Average the LSTM outputs across all frames.
        # Shape: [batch, hidden_dim]
        video_features = temporal_features.mean(dim=1)

        # Produce one raw score for every video.
        # Initial shape: [batch, 1]
        logits = self.classifier(video_features)

        # Remove the final size-1 dimension.
        # Final shape: [batch]
        logits = logits.squeeze(1)

        return logits

In [ ]:
model = BinaryVideoClassifier()

criterion = nn.BCEWithLogitsLoss() #Expects raw Logits - no function

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4,
)

videos = videos.to(device)                  # float: [B, T, 3, H, W]
labels = labels.to(device).float()          # float: [B]

logits = model(videos)                      # [B]
loss = criterion(logits, labels)

optimizer.zero_grad()
loss.backward()
optimizer.step()

In [ ]:
logits = model(videos) #forward method of model
loss = criterion(logits, labels)

Loss and Training